# 01 · ¿Una tienda con competidor a 2 km tiene precios distintos?

**Objetivo:** Mostrar a directores y dueños si la competencia cercana cambia el precio de una tienda, como caso de competencia local para decisiones de expansión y de precio por tienda.

**Pregunta:** Con la misma cadena y en la misma ciudad, ¿cuánto difiere el índice de precio de las tiendas con y sin competidor a 2 km?

Importa porque decide si conviene fijar precios por tienda según la competencia local o mantener un precio por
zona. **Diferencia** = cuánto más (o menos) cobra, en promedio, una tienda con competidor cerca que otra de la misma cadena
y ciudad sin competidor. IC90 = rango probable (bootstrap de ciudades).

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import competencia as K
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
REPS = 500
foc = pd.read_csv(PROC / "tiendas_focales.csv")
ev = foc[foc.grupo.isin(K.EVALUABLES)].copy()
tot = K.estimate(ev, K.comp(), reps=REPS)
byg = K.estimate(ev, K.comp_by_group(), reps=REPS)
res_ = pd.concat([tot.rename(index={"competidor": "Walmart de México y Soriana"}),
                  byg.rename(index=lambda s: s.split(" · ")[1])])
display(res_.round(2))
for k_, r_ in res_.iterrows():
    assert r_.ic90_inf <= r_.diferencia_pct <= r_.ic90_sup, k_

,diferencia_pct,ic90_inf,ic90_sup,solido
Walmart de México y Soriana,0.35,-0.16,0.88,False
Walmart de México,0.26,-0.21,0.74,False
Soriana,0.54,-0.46,1.77,False


## Sensibilidad

- incluir las tiendas de Chedraui en la regresión (su efecto no se reporta);
- solo tiendas con 300 SKUs o más en el índice;
- solo ciudades con tiendas de los dos tipos;
- controlar por grupo corporativo en lugar de formato.

In [2]:
both = ev.groupby("municipio_id").apply(lambda z: (z.dmin <= 2).any() and (z.dmin > 2).any(), include_groups=False)
var = {"Principal": ev, "Con Chedraui en la regresión": foc, "Tiendas con 300+ SKUs": ev[ev.skus >= 300],
       "Solo ciudades con ambos tipos": ev[ev.municipio_id.isin(both[both].index)],
       "Efecto fijo de grupo": ev.assign(formato=ev.grupo)}
sens = pd.DataFrame({k: K.estimate(v, K.comp(), reps=200).iloc[0] for k, v in var.items()}).T
sens["tiendas"] = [len(v) for v in var.values()]
display(sens.round(2))

,diferencia_pct,ic90_inf,ic90_sup,solido,tiendas
Principal,0.345976,-0.150801,0.911866,False,217
Con Chedraui en la regresión,0.327766,-0.113306,0.898051,False,247
Tiendas con 300+ SKUs,0.044262,-0.338842,0.434823,False,196
Solo ciudades con ambos tipos,0.353842,-0.048359,0.844841,False,128
Efecto fijo de grupo,0.568227,-0.292598,1.676613,False,217


In [3]:
res_.to_csv(PROC / "efecto_2km.csv", encoding="utf-8-sig")
sens.to_csv(PROC / "sensibilidad_2km.csv", encoding="utf-8-sig")
F = "01_efecto_a_2km"
for k_, kk in {"Walmart de México y Soriana": "total", "Walmart de México": "walmex", "Soriana": "soriana"}.items():
    r_ = res_.loc[k_]
    reg(f"ef_{kk}", r_.diferencia_pct, [r_.ic90_inf, r_.ic90_sup], n=len(ev) if kk == "total" else (ev.grupo == k_).sum(), fuente=F)
reg("sens_min", sens.diferencia_pct.min(), fuente=F); reg("sens_max", sens.diferencia_pct.max(), fuente=F)
reg("sens_solidas", int(sens.solido.sum()), n=len(sens), fuente=F)
reg("tiendas_ev", len(ev), fuente=F)
t = res_.loc["Walmart de México y Soriana"]
display(Markdown(
    f"**Conclusión.** Con la misma cadena y en la misma ciudad, una tienda con competidor a 2 km o menos tiene un índice **{t.diferencia_pct:+.2f}%** "
    f"frente a una sin competidor (IC90 {t.ic90_inf:+.2f} a {t.ic90_sup:+.2f}%; {len(ev)} tiendas). No es más barata: la diferencia es de "
    f"décimas de punto y, si acaso, hacia arriba. En las sensibilidades va de {sens.diferencia_pct.min():+.2f}% a {sens.diferencia_pct.max():+.2f}%."))

**Conclusión.** Con la misma cadena y en la misma ciudad, una tienda con competidor a 2 km o menos tiene un índice **+0.35%** frente a una sin competidor (IC90 -0.16 a +0.88%; 217 tiendas). No es más barata: la diferencia es de décimas de punto y, si acaso, hacia arriba. En las sensibilidades va de +0.04% a +0.57%.